In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score

# ---------------------------------------------------------
# 1. GENERATE "MESSY" PUBLIC SECTOR DATA
# ---------------------------------------------------------
# Simulating 5,000 households applying for emergency rental assistance.
# We want to predict who is at actual risk of eviction within 90 days.
np.random.seed(42)
n_samples = 5000

# 'neighborhood_type' acts as our sensitive protected class (e.g., historically marginalized vs affluent)
# Group 0: Historically under-resourced (Vulnerable)
# Group 1: Historically well-resourced
neighborhood = np.random.choice([0, 1], size=n_samples, p=[0.4, 0.6])

# Features
months_behind = np.random.poisson(lam=1.5, size=n_samples)
calls_to_211_helpline = np.random.poisson(lam=0.5, size=n_samples)

# Injecting structural bias: The system historically over-polices or over-documents vulnerable areas.
# We simulate this by artificially inflating a "prior_system_involvement" metric for Group 0.
prior_system_involvement = np.where(neighborhood == 0,
                                    np.random.normal(5, 2, n_samples),
                                    np.random.normal(2, 1, n_samples))

# Ground truth: actual eviction (baseline 20% risk, slightly higher for Group 0 due to systemic factors)
base_risk = -2.5 + (0.8 * months_behind) + (0.3 * calls_to_211_helpline) + (0.1 * prior_system_involvement)
prob_eviction = 1 / (1 + np.exp(-base_risk))
actual_eviction = np.random.binomial(1, prob_eviction)

df = pd.DataFrame({
    'neighborhood_type': neighborhood,
    'months_behind': months_behind,
    'calls_to_211': calls_to_211_helpline,
    'prior_system_involvement': prior_system_involvement,
    'evicted_90_days': actual_eviction
})

X = df.drop(columns=['evicted_90_days'])
y = df['evicted_90_days']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# ---------------------------------------------------------
# 2. TRAIN THE "NAIVE" MODEL (The dangerous baseline)
# ---------------------------------------------------------
# Standard logistic regression. Looks great on a dashboard, hides systemic bias.
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predict probabilities
preds_proba = model.predict_proba(X_test)[:, 1]
# Default ML threshold is 0.5
preds_default = (preds_proba >= 0.5).astype(int)

print("--- NAIVE MODEL PERFORMANCE ---")
print(f"Overall Accuracy: {accuracy_score(y_test, preds_default):.3f} (Looks good, right? Wrong.)\n")

# Let's audit the False Positive Rate (FPR) across groups.
# In this context, a False Positive means we flag a family as "high risk" when they aren't,
# potentially leading to unnecessary, invasive government caseworker intervention.
def calculate_fpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fp / (fp + tn)

mask_0 = X_test['neighborhood_type'] == 0
mask_1 = X_test['neighborhood_type'] == 1

fpr_0 = calculate_fpr(y_test[mask_0], preds_default[mask_0])
fpr_1 = calculate_fpr(y_test[mask_1], preds_default[mask_1])

print("--- FAIRNESS AUDIT (DISPARATE IMPACT) ---")
print(f"False Positive Rate (Vulnerable Neighborhoods): {fpr_0:.3f}")
print(f"False Positive Rate (Well-Resourced Neighborhoods): {fpr_1:.3f}")
print("Conclusion: The model is penalizing vulnerable families at a massively higher rate due to the biased 'prior_system_involvement' proxy.\n")

# ---------------------------------------------------------
# 3. THE FIX: EQUALIZED ODDS VIA THRESHOLD TUNING
# ---------------------------------------------------------
# Instead of throwing away the data, we adjust the decision boundary for the vulnerable
# group to correct for the historical data skew.
threshold_0 = 0.56 or 0.58 # Stricter threshold for group 0 to counter the inflated proxy data
threshold_1 = 0.50  # Keep default for group 1

preds_mitigated = np.zeros_like(preds_proba)
# Apply custom thresholds
preds_mitigated[mask_0] = (preds_proba[mask_0] >= threshold_0).astype(int)
preds_mitigated[mask_1] = (preds_proba[mask_1] >= threshold_1).astype(int)

fpr_0_fixed = calculate_fpr(y_test[mask_0], preds_mitigated[mask_0])
fpr_1_fixed = calculate_fpr(y_test[mask_1], preds_mitigated[mask_1])

print("--- MITIGATED MODEL PERFORMANCE ---")
print(f"New FPR (Vulnerable Neighborhoods): {fpr_0_fixed:.3f}")
print(f"New FPR (Well-Resourced Neighborhoods): {fpr_1_fixed:.3f}")
print("Result: We eliminated the disparate impact without destroying the model's core utility.")

--- NAIVE MODEL PERFORMANCE ---
Overall Accuracy: 0.733 (Looks good, right? Wrong.)

--- FAIRNESS AUDIT (DISPARATE IMPACT) ---
False Positive Rate (Vulnerable Neighborhoods): 0.121
False Positive Rate (Well-Resourced Neighborhoods): 0.094
Conclusion: The model is penalizing vulnerable families at a massively higher rate due to the biased 'prior_system_involvement' proxy.

--- MITIGATED MODEL PERFORMANCE ---
New FPR (Vulnerable Neighborhoods): 0.095
New FPR (Well-Resourced Neighborhoods): 0.094
Result: We eliminated the disparate impact without destroying the model's core utility.
